# Lecture 6: Threshold Effect and Evaluation Figures

Interactive demo for threshold behavior and confusion-matrix-style outcomes.
Use the threshold slider to see precision/recall and TP/FP/TN/FN update.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

rng = np.random.default_rng(7)


def make_data(n=24, noise=0.12):
    y = np.array([1]*12 + [0]*12)
    base_pos = rng.normal(0.75, noise, 12)
    base_neg = rng.normal(0.35, noise, 12)
    s = np.clip(np.concatenate([base_pos, base_neg]), 0.0, 1.0)
    return s, y

scores, y_true = make_data()


def metrics_at_threshold(t=0.5):
    y_pred = (scores >= t).astype(int)
    tp = int(((y_true==1)&(y_pred==1)).sum())
    fp = int(((y_true==0)&(y_pred==1)).sum())
    tn = int(((y_true==0)&(y_pred==0)).sum())
    fn = int(((y_true==1)&(y_pred==0)).sum())
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    return tp, fp, tn, fn, p, r, y_pred


def plot_threshold(threshold=0.5):
    tp, fp, tn, fn, p, r, y_pred = metrics_at_threshold(threshold)

    ths = np.linspace(0.01, 0.99, 80)
    prec, rec = [], []
    for t in ths:
        tp_, fp_, tn_, fn_, p_, r_, _ = metrics_at_threshold(t)
        prec.append(p_)
        rec.append(r_)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), gridspec_kw={'height_ratios':[1.0,1.1]})
    ax0, ax1, ax2, ax3 = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

    order = np.argsort(scores)
    ax0.scatter(np.arange(len(scores)), scores[order], c=y_true[order], cmap='coolwarm', s=45)
    ax0.axhline(threshold, color='black', ls='--', lw=1.8)
    ax0.set_title('Sorted scores with threshold')
    ax0.set_xlabel('Sorted sample index')
    ax0.set_ylabel('Score')

    cm = np.array([[tp, fn],[fp, tn]])
    im = ax1.imshow(cm, cmap='Blues', vmin=0)
    for i in range(2):
        for j in range(2):
            ax1.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14, color='black')
    ax1.set_xticks([0,1], labels=['Pred +','Pred -'])
    ax1.set_yticks([0,1], labels=['True +','True -'])
    ax1.set_title('Confusion matrix view')

    ax2.plot(ths, prec, 'b--', lw=2, label='Precision')
    ax2.plot(ths, rec, 'g-', lw=2, label='Recall')
    ax2.axvline(threshold, color='k', ls=':', lw=1.5)
    ax2.set_ylim(-0.02, 1.02)
    ax2.set_xlabel('Threshold')
    ax2.set_ylabel('Metric value')
    ax2.set_title('Precision / Recall trade-off')
    ax2.legend()

    ax3.axis('off')
    txt = (
        f'Threshold: {threshold:.2f}

'
        f'TP={tp}, FP={fp}, TN={tn}, FN={fn}
'
        f'Precision={p:.3f}
Recall={r:.3f}'
    )
    ax3.text(0.05, 0.55, txt, fontsize=13)

    plt.tight_layout()
    plt.show()

slider = widgets.FloatSlider(description='threshold', min=0.0, max=1.0, step=0.01, value=0.5, continuous_update=False)
regen = widgets.Button(description='Regenerate data', button_style='')
out = widgets.Output()

def redraw(*_):
    with out:
        out.clear_output(wait=True)
        plot_threshold(slider.value)

def on_regen(_):
    global scores, y_true
    scores, y_true = make_data()
    redraw()

slider.observe(redraw, names='value')
regen.on_click(on_regen)
display(widgets.HBox([slider, regen]), out)
redraw()
